# Capítulo 10 · Laboratório 10: osciladores de momentum
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap10.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Os osciladores do capítulo escritos à mão, com pandas: MACD, MACD-V, momento, ROC, KST, Coppock, Trix, Elder-Ray, IFR, estocástico, Williams %R e CCI. O laboratório confere as contas do texto, compara o MACD com o MACD-V de Alex Spiroglou, testa as faixas de IFR em tendência, a sobrecompra em alta, o CCI no zero contra o +100, a Curva de Coppock desde 1951, o RAFI didático e o ranking de momentum de Clenow.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap10.pkl"
pc = lambda x, k=1: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=2: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
ATIVOS = {"Ibovespa": "^BVSP", "PETR4": "PETR4.SA", "VALE3": "VALE3.SA", "ITUB4": "ITUB4.SA", "WEGE3": "WEGE3.SA",
          "ABEV3": "ABEV3.SA", "GGBR4": "GGBR4.SA", "Bitcoin": "BTC-USD", "Ether": "ETH-USD", "S&P 500": "^GSPC",
          "Dow Jones": "^DJI", "Microsoft": "MSFT"}
ACOES = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
         "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "BPAC11", "SBSP3", "CSAN3"]


def baixa(t, ini):
    d = yf.download(t, start=ini, end=FIM, auto_adjust=True, progress=False)
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    return d.dropna(subset=["Close"])


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


D = pd.read_pickle(CACHE) if os.path.exists(CACHE) else {}
for nome, t in ATIVOS.items():
    if nome not in D:
        D[nome] = baixa(t, "1950-01-01" if nome == "S&P 500" else "2000-01-01")
confere({nome: D[nome] for nome in ATIVOS})
for t in ACOES:
    if t not in D:
        D[t] = baixa(f"{t}.SA", "2024-01-01")
pd.to_pickle(D, CACHE)

## Parte 1: dados

In [ ]:
def limpa(d, volume=True):
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"])
    if volume and (d["Volume"] > 0).mean() > 0.5:
        ok &= d["Volume"] > 0
    return d[ok].copy()


A = {k: limpa(D[k], not k.startswith(("Ibov", "S&P", "Dow"))) for k in ATIVOS}
print(f"Parte 1. {len(A)} ativos (Ibovespa desde {A['Ibovespa'].index[0]:%Y}; S&P 500 desde"
      f" {A['S&P 500'].index[0]:%Y}); último pregão {A['Ibovespa'].index[-1]:%d/%m/%Y}.")

## Parte 2: os osciladores, escritos à mão

In [ ]:
def mme(x, n):
    return x.ewm(span=n, adjust=False).mean()


def macd(c, r=12, l=26, s=9):
    linha = mme(c, r) - mme(c, l)
    sinal = mme(linha, s)                                  # a linha de sinal é a MME de 9 da linha do MACD
    return linha, sinal, linha - sinal


def true_range(d):
    return pd.concat([d["High"] - d["Low"], (d["High"] - d["Close"].shift()).abs(),
                      (d["Low"] - d["Close"].shift()).abs()], axis=1).max(axis=1)


def atr(d, n):
    return true_range(d).ewm(alpha=1 / n, adjust=False).mean()


def macd_v(d):
    """MACD-V de Alex Spiroglou: [MME(12) - MME(26)] / ATR(26) x 100."""
    return (mme(d["Close"], 12) - mme(d["Close"], 26)) / atr(d, 26) * 100


def momento(c, n=10):
    return c - c.shift(n)                                  # preço de hoje menos o de n períodos atrás


def roc(c, n=20):
    return (c / c.shift(n) - 1) * 100


def kst(c, r=(10, 15, 20, 30), m=(10, 10, 10, 15), s=9):
    """KST de Martin Pring: quatro ROC suavizados, pesos 1 a 4, e linha de sinal de 9."""
    k = sum((i + 1) * roc(c, a).rolling(b).mean() for i, (a, b) in enumerate(zip(r, m)))
    return k, k.rolling(s).mean()


def mmp(x, n):
    pesos = np.arange(1, n + 1)
    return x.rolling(n).apply(lambda v: np.dot(v, pesos) / pesos.sum(), raw=True)


def coppock(cm):
    """Curva de Coppock em dados mensais: média ponderada de 10 meses de (ROC de 14 + ROC de 11)."""
    return mmp(roc(cm, 14) + roc(cm, 11), 10)


def trix(c, n=15, s=9):
    t = mme(mme(mme(c, n), n), n)
    tr = t.pct_change() * 100
    return tr, mme(tr, s)


def elder(d, n=13):
    m = mme(d["Close"], n)
    return d["High"] - m, d["Low"] - m, m                 # bull power, bear power e a média


def ifr(c, n=14):
    delta = c.diff()
    g = delta.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    p = (-delta.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    return 100 * g / (g + p)                               # o mesmo que 100 - 100 / (1 + g / p)


def estocastico(d, n=14, k=3, dd=3):
    hh, ll = d["High"].rolling(n).max(), d["Low"].rolling(n).min()
    rapido = 100 * (d["Close"] - ll) / (hh - ll)           # %K rápido
    lento = rapido.rolling(k).mean()                       # %K lento = %D rápido
    return rapido, lento, lento.rolling(dd).mean()          # rápido, lento e %D lento


def williams(d, n=14):
    hh, ll = d["High"].rolling(n).max(), d["Low"].rolling(n).min()
    return -100 * (hh - d["Close"]) / (hh - ll)


def cci(d, n=20):
    """CCI de Lambert: (preço típico - média) / (0,015 x desvio médio absoluto), e não o desvio-padrão."""
    pt = (d["High"] + d["Low"] + d["Close"]) / 3
    dm = pt.rolling(n).apply(lambda v: np.mean(np.abs(v - v.mean())), raw=True)
    return (pt - pt.rolling(n).mean()) / (0.015 * dm)


def adx(d, n=14):
    sobe, desce = d["High"].diff(), -d["Low"].diff()
    mais = pd.Series(np.where((sobe > desce) & (sobe > 0), sobe, 0.0), d.index)
    menos = pd.Series(np.where((desce > sobe) & (desce > 0), desce, 0.0), d.index)
    a = atr(d, n)
    pdi = 100 * mais.ewm(alpha=1 / n, adjust=False).mean() / a
    ndi = 100 * menos.ewm(alpha=1 / n, adjust=False).mean() / a
    return (100 * (pdi - ndi).abs() / (pdi + ndi)).ewm(alpha=1 / n, adjust=False).mean(), pdi, ndi


def clenow(c, n=90):
    """Momentum de Andreas Clenow: inclinação anualizada da regressão do log do preço (90 dias) x R²."""
    y = np.log(c.iloc[-n:].to_numpy())
    x = np.arange(n)
    b, _ = np.polyfit(x, y, 1)
    return (np.exp(b * 252) - 1) * 100 * np.corrcoef(x, y)[0, 1] ** 2

## Parte 3: as contas do texto

In [ ]:
P = A["PETR4"]
c = P["Close"]
g = c.diff().clip(lower=0).ewm(alpha=1 / 14, adjust=False).mean()
p = (-c.diff().clip(upper=0)).ewm(alpha=1 / 14, adjust=False).mean()
print("\nParte 3. As contas do texto")
print(f"   IFR pelas duas fórmulas, PETR4 em {c.index[-1]:%d/%m/%Y}: 100 - 100/(1 + RS) = {nr(100 - 100 / (1 + g / p).iloc[-1])}"
      f" | 100 x A/(A + B) = {nr(ifr(c).iloc[-1])}")
rap, _, _ = estocastico(P)
print(f"   Williams %R = %K rápido - 100: diferença máxima {nr((williams(P) - (rap - 100)).abs().max(), 10)}")
DENTRO_CCI = {}
for n in (10, 20, 40):
    v = pd.concat([cci(A[k], n) for k in ("Ibovespa", "PETR4", "VALE3", "ITUB4")]).dropna()
    DENTRO_CCI[n] = (v.abs() <= 100).mean()
print("   CCI entre -100 e +100 (Ibovespa, PETR4, VALE3 e ITUB4): " + " | ".join(
    f"{n} períodos {pc(x)}" for n, x in DENTRO_CCI.items()))

## Parte 4: MACD x MACD-V

In [ ]:
TAB_V = []
for nome in ("Ibovespa", "PETR4", "VALE3", "ITUB4", "WEGE3", "Bitcoin", "S&P 500"):
    d = A[nome].loc["2015":]
    m, _, _ = macd(d["Close"])
    mv = macd_v(d)
    m, mv = m.iloc[30:], mv.iloc[30:]
    TAB_V.append((nome, m.abs().quantile(0.95), mv.abs().quantile(0.95), (mv.abs() <= 50).mean(), (mv > 150).mean(),
                  (mv < -150).mean()))
print("\nParte 4. 95% dos dias dentro de (±), desde 2015: MACD em pontos de preço x MACD-V")
for nome, a95, v95, lat, alto, baixo in TAB_V:
    print(f"   {nome:<9} MACD {nr(a95):>10} | MACD-V {v95:>4.0f}")

In [ ]:
import matplotlib.pyplot as plt
fig, axs = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
for nome in ("Ibovespa", "PETR4", "Bitcoin"):
    d = A[nome].iloc[-500:]
    axs[0].plot(d.index, macd(A[nome]["Close"])[0].loc[d.index] / A[nome]["Close"].loc[d.index].iloc[-1], lw=0.8, label=nome)
    axs[1].plot(d.index, macd_v(A[nome]).loc[d.index], lw=0.8, label=nome)
axs[0].set_title("MACD dividido pelo último preço (só para caber no mesmo gráfico)")
axs[1].set_title("MACD-V: a mesma régua para todos")
for y in (-150, -50, 50, 150):
    axs[1].axhline(y, color="gray", lw=0.5, ls="--")
axs[1].legend()
plt.show()

## Parte 5: lateralidade

In [ ]:
print("\nParte 5. Dias com o MACD-V entre -50 e +50 (zona de lateralidade), desde 2015")
for nome, a95, v95, lat, alto, baixo in TAB_V:
    print(f"   {nome:<9} {pc(lat, 0)} | acima de +150 {pc(alto, 0)} | abaixo de -150 {pc(baixo, 0)}")
sp = A["S&P 500"].loc["1975":"2021"]
LAT_SP = (macd_v(sp).iloc[30:].abs() <= 50).mean()
print(f"   S&P 500 de 1975 a 2021 (o período do artigo): {pc(LAT_SP, 0)}")

## Parte 6: faixas de IFR

In [ ]:
FAIXAS = {}
print("\nParte 6. IFR(14) com o preço acima e abaixo da média de 200 (desde 2000)")
for nome in ("Ibovespa", "PETR4", "VALE3", "S&P 500"):
    c = A[nome]["Close"].loc["2000":]
    r, mm = ifr(c), c.rolling(200).mean()
    ok = mm.notna()
    ra, rb = r[(c > mm) & ok], r[(c <= mm) & ok]
    FAIXAS[nome] = (ra.quantile(.1), ra.quantile(.9), (ra < 40).mean(), rb.quantile(.1), rb.quantile(.9), (rb > 60).mean(),
                    ra, rb)
    print(f"   {nome:<9} alta: 80% dos dias entre {ra.quantile(.1):.0f} e {ra.quantile(.9):.0f} (abaixo de 40 em"
          f" {pc((ra < 40).mean(), 0)}) | baixa: entre {rb.quantile(.1):.0f} e {rb.quantile(.9):.0f} (acima de 60 em"
          f" {pc((rb > 60).mean(), 0)})")

## Parte 7: sobrecompra em tendência

In [ ]:
SOBRE = {}
print("\nParte 7. IFR(14) acima de 70 com o preço acima da média de 200: retorno dos 20 pregões seguintes")
for nome in ("Ibovespa", "PETR4", "VALE3", "S&P 500"):
    c = A[nome]["Close"].loc["2000":]
    r, mm = ifr(c), c.rolling(200).mean()
    f20 = c.shift(-20) / c - 1
    alta = (c > mm) & mm.notna()
    s, t = f20[alta & (r > 70)].dropna(), f20[alta].dropna()
    SOBRE[nome] = (len(s), s.mean(), (s > 0).mean(), t.mean(), (t > 0).mean())
    print(f"   {nome:<9} sobrecomprado: {len(s):>4} dias, {ps(s.mean())} (positivos {pc((s > 0).mean(), 0)}) | qualquer dia"
          f" em alta: {ps(t.mean())} (positivos {pc((t > 0).mean(), 0)})")

## Parte 8: CCI no zero x CCI em +100

In [ ]:
def estado(entra, sai):
    pos, lista = 0, []
    for e, s in zip(entra.to_numpy(), sai.to_numpy()):
        if pos == 0 and e:
            pos = 1
        elif pos == 1 and s:
            pos = 0
        lista.append(pos)
    return pd.Series(lista, entra.index)


def resultado(c, pos, custo=0.001):
    trocas = pos.diff().abs().fillna(0)
    ret = c.pct_change().fillna(0) * pos.shift().fillna(0) - trocas * custo / 2
    curva = (1 + ret).cumprod()
    return curva.iloc[-1] ** (252 / len(c)) - 1, (curva / curva.cummax() - 1).min(), int((pos.diff() == 1).sum()), pos.mean()


CCI_T = {}
print("\nParte 8. CCI(20) só comprado, 2010 a 2026, custo de 0,10% por ida e volta")
for nome in ("Ibovespa", "PETR4", "VALE3", "ITUB4"):
    d = A[nome].loc["2010":]
    c, k = d["Close"], cci(d)
    regras = {"zero": estado((k > 0) & (k.shift() <= 0), (k < 0) & (k.shift() >= 0)),
              "+100": estado((k > 100) & (k.shift() <= 100), (k < 100) & (k.shift() >= 100)),
              "segurar": pd.Series(1, c.index)}
    CCI_T[nome] = {r: resultado(c, v) for r, v in regras.items()}
    print(f"   {nome:<9} " + " | ".join(f"{r}: {pc(v[0])} ao ano, queda {pc(v[1], 0)}, {v[2]} op."
                                       for r, v in CCI_T[nome].items()))

## Parte 9: Curva de Coppock

In [ ]:
sm = A["S&P 500"]["Close"].resample("ME").last()
CP = coppock(sm)
compra = CP[(CP > 0) & (CP.shift() <= 0)].index
f12 = sm.shift(-12) / sm - 1
COPP = pd.DataFrame({"data": compra, "r12": f12.reindex(compra).to_numpy()}).dropna()
print(f"\nParte 9. Coppock no S&P 500 (mensal, desde 1951): {len(COPP)} sinais de compra (cruzou o zero para cima)")
print(f"   retorno médio em 12 meses: {ps(COPP['r12'].mean())} (positivos {pc((COPP['r12'] > 0).mean(), 0)}) | qualquer mês:"
      f" {ps(f12.dropna().mean())} (positivos {pc((f12.dropna() > 0).mean(), 0)})")
print("   últimos sinais: " + ", ".join(f"{x:%m/%Y}" for x in compra[-5:]))

## Parte 10: RAFI didático

In [ ]:
RAFI = {}
print("\nParte 10. RAFI didático (ADX de 8 de hoje menos o de ontem), dentro de tendências de alta (+DI > -DI e ADX > 20)")
print("   Retorno médio dos 10 pregões seguintes, com e sem o 'chute' (RAFI abaixo de -2,5)")
for nome in ("Ibovespa", "PETR4", "VALE3", "ITUB4"):
    d = A[nome].loc["2010":]
    c = d["Close"]
    a8, pdi, ndi = adx(d, 8)
    rafi, alta = a8.diff(), (pdi > ndi) & (a8 > 20)
    f = c.shift(-10) / c - 1
    k, o = f[alta & (rafi < -2.5)].dropna(), f[alta & (rafi >= -2.5)].dropna()
    RAFI[nome] = (len(k), k.mean(), (k > 0).mean(), len(o), o.mean(), (o > 0).mean())
    print(f"   {nome:<9} com chute: {len(k):>4} dias, {ps(k.mean())} (positivos {pc((k > 0).mean(), 0)}) | sem chute:"
          f" {len(o):>4} dias, {ps(o.mean())} (positivos {pc((o > 0).mean(), 0)})")

## Parte 11: momentum de Clenow

In [ ]:
RANK = []
for t in ACOES:
    c = limpa(D[t])["Close"]
    if len(c) >= 90:
        RANK.append((t, clenow(c)))
RANK.sort(key=lambda x: -x[1])
print(f"\nParte 11. Momentum de Clenow (90 pregões até {c.index[-1]:%d/%m/%Y}), 20 ações:")
for t, v in RANK:
    print(f"   {t:<7} {nr(v, 1):>7}")

In [ ]:
import matplotlib.pyplot as plt
nomes, vals = zip(*RANK[::-1])
plt.figure(figsize=(7, 5))
plt.barh(nomes, vals, color=["gray" if v >= 0 else "black" for v in vals])
plt.title("Momentum de Clenow (90 pregões x R²)")
plt.show()

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `ifr` da Parte 2. Mostre onde está a suavização de Wilder (alpha = 1/14) e por que `100 * g / (g + p)` é igual a `100 - 100 / (1 + g / p)`." Confira com a Parte 3, que calcula o IFR pelas duas fórmulas.
2. **Adaptar:** "Acrescente [ATIVO] ao dicionário `ATIVOS` da célula de abertura, com o ticker do Yahoo Finance, e inclua o nome dele na lista da Parte 6. Não mude a fórmula do IFR nem a média de 200." Antes de olhar o ativo novo, confira se os números do Ibovespa continuam iguais aos do livro.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 6, troque a média de 200 pela de 50. As faixas de IFR mudam?
2. Na Parte 8, teste o CCI de 40 períodos. O zero continua rendendo mais que o +100?
3. Na Parte 10, mude o prazo de 10 para 5 pregões. O 'chute' do RAFI fica mais claro?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.